In [26]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertForSequenceClassification,
    pipeline
)

import warnings
import logging
import torch.nn.functional as F
import time
import numpy as np
import re

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

In [20]:
base_dir = os.getcwd()
print("Project base directory:", base_dir)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

Project base directory: C:\Users\Bill Yin\Documents\GitHub\Strategy Development\NLP Equity L-S
Using device: cuda


In [36]:
def add_exchange_suffix(code: str) -> str:
    code = str(code)

    if len(code) == 5 and code.isdigit():
        ticker = f"{code}.HK"
        return ticker

    if code.startswith(("60", "68", "900")):
        ticker = f"{code}.SH"
        return ticker

    if code.startswith(("00", "30", "200")):
        ticker = f"{code}.SZ"
        return ticker

    return code


def sentence_split(text):
    # Split on 。！？； keep punctuation
    parts = re.split(r'([。！？；])', text)
    sentences = ["".join([parts[i], parts[i+1]]) for i in range(0, len(parts)-1, 2)]
    return [s.strip() for s in sentences if s.strip()]


def translate_sentence_by_sentence(text, translator, max_length=256):
    """
    Translate text sentence by sentence using the given translator.
    """
    if text is None:
        return None
    sentences = sentence_split(text)
    translated = []
    for s in sentences:
        if s:
            en = translator(s, max_length=max_length, truncation=True)[0]["translation_text"]
            translated.append(en)
    return " ".join(translated).strip()


def text_pre_process(text: str) -> str:
    """
    Preprocess Chinese financial text before translation:
    - Normalize punctuation
    - Replace financial units with English equivalents
    - Protect numbers with placeholders
    - Clean up formatting
    """

    if text is None or not isinstance(text, str):
        return text, {}

    # --- Normalize punctuation ---
    text = text.replace("（", "(").replace("）", ")")
    text = text.replace("，", ", ").replace("：", ": ")
    text = text.replace("；", "; ").replace("。", ". ")

    # --- Replace financial units ---
    text = text.replace("人民币", "RMB")
    text = text.replace("亿元", " billion RMB")
    text = text.replace("百万元", " million RMB")
    text = text.replace("万元", " ten-thousand RMB")
    text = text.replace("元/股", " RMB/share")
    text = text.replace("元", " RMB")
    text = text.replace("股", " shares")

    # --- Protect numbers & percentages with placeholders ---
    numbers = re.findall(r"\d+(?:\.\d+)?%?", text)
    placeholder_map = {}
    for i, num in enumerate(numbers):
        placeholder = f"⟦{i}⟧"
        placeholder_map[placeholder] = num
        text = text.replace(num, placeholder, 1)

    # --- Clean spaces ---
    text = re.sub(r"\s+", " ", text).strip()

    return text, placeholder_map


def restore_numbers(translated_text: str, placeholder_map: dict) -> str:
    """
    Restore original numbers into translated text
    """
    if translated_text is None:
        return None

    for placeholder, num in placeholder_map.items():
        # Escape special chars like ⟦0⟧
        translated_text = re.sub(re.escape(placeholder), num, translated_text, flags=re.IGNORECASE)

    return translated_text


def process_report(ID, title_cn, summary_cn, pre_processing=False):
    if pre_processing:
        title_cn, title_map = text_pre_process(title_cn)
        summary_cn, summary_map = text_pre_process(summary_cn)
    else:
        title_map, summary_map = {}, {}

    # --- Translation ---
    title_en = translator(title_cn, max_length=600, truncation=True)[0]["translation_text"] if pd.notna(title_cn) else None
    summary_en = translate_sentence_by_sentence(summary_cn, translator, max_length=256) if pd.notna(summary_cn) else None

    # --- Restore numbers if preprocessed ---
    if pre_processing:
        title_en = restore_numbers(title_en, title_map)
        summary_en = restore_numbers(summary_en, summary_map)

    # --- Title classification ---
    if title_en is not None:
        inputs = tokenizer(title_en, return_tensors="pt", truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            title_logits = model(**inputs).logits.squeeze().cpu().numpy()
            title_probs = F.softmax(torch.tensor(title_logits), dim=-1).numpy()
    else:
        title_logits = [None, None, None]
        title_probs = [None, None, None]

    # --- Summary classification ---
    if summary_en is not None:
        inputs = tokenizer(summary_en, return_tensors="pt", truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            summary_logits = model(**inputs).logits.squeeze().cpu().numpy()
            summary_probs = F.softmax(torch.tensor(summary_logits), dim=-1).numpy()
    else:
        summary_logits = [None, None, None]
        summary_probs = [None, None, None]

    # --- Build outputs ---
    translation_dict = {
        "REPORT_ID": ID,
        "TITLE_EN": title_en,
        "SUMMARY_EN": summary_en
    }

    logit_dict = {
        "REPORT_ID": ID,
        "title_neg_logit": title_logits[0],
        "title_neu_logit": title_logits[1],
        "title_pos_logit": title_logits[2],
        "summary_neg_logit": summary_logits[0],
        "summary_neu_logit": summary_logits[1],
        "summary_pos_logit": summary_logits[2],
    }

    softmax_dict = {
        "REPORT_ID": ID,
        "title_neg_prob": title_probs[0],
        "title_neu_prob": title_probs[1],
        "title_pos_prob": title_probs[2],
        "summary_neg_prob": summary_probs[0],
        "summary_neu_prob": summary_probs[1],
        "summary_pos_prob": summary_probs[2],
    }

    return translation_dict, logit_dict, softmax_dict

In [17]:
equity_reports = pd.read_csv(
    os.path.join(base_dir, "equity_reports_1.csv"),
    encoding="utf-8",
    parse_dates=["PUBLISH_DATE", "UPDATE_TIME", "UPDATE_DATE"],
    dtype={"STOCK_CODE": str}
)

print("Loaded equity_reports with shape:", equity_reports.shape)

equity_reports = equity_reports.dropna(subset=["STOCK_CODE", "REPORT_ID"])
equity_reports["REPORT_ID"] = equity_reports["REPORT_ID"].astype(int)
equity_reports["STOCK_CODE"] = equity_reports["STOCK_CODE"].apply(add_exchange_suffix)

equity_reports.head()

Loaded equity_reports with shape: (10874, 9)


,REPORT_ID,SUMMARY,TITLE,PUBLISH_DATE,STOCK_CODE,STOCK_ABBR,UPDATE_TIME,UPDATE_DATE,Lag
0,5084530,巴比食品(605338)事件：公司发布《2022年限制性股票激励计划》，拟向激励对象授予的限...,巴比食品（605338）：业绩目标稳健，完善激励机制,2022-12-31,605338.SH,巴比食品,2023-01-02 15:44:24,2023-01-02,2
1,5084531,创新纳米给药系统（Drug Delivery System）的高壁垒领先胶束平台。（1）\n...,上海谊众（688091）：全球纳米新秀，国产独家紫杉醇胶束重磅上市,2022-12-31,688091.SH,上海谊众,2023-01-02 16:36:49,2023-01-02,2
2,5084533,事件：1、公司近日公告拟在浙江桐乡投资设立“浙江巨石新能源有限公司”（暂定名），该公司将作为...,中国巨石（600176）：布局新能源零碳基地，竞争力再夯实,2022-12-31,600176.SH,中国巨石,2023-01-02 15:44:03,2023-01-02,2
3,5084535,调整组织架构，有望加强资源协同降本增效为进一步规范公司治理、完善公司职能、优化资源配置，更好...,太平鸟（603877）：调整组织架构降本增效，夯实内功静待终端复苏,2022-12-31,603877.SH,太平鸟,2023-01-03 10:19:18,2023-01-03,3
4,5084545,事件：公司于2022年12月29日收盘后发布《第三期限制性股票计划（草案）》等公告。点评：发...,宝信软件（600845）：股权激励计划落地，长期持续成长无忧,2022-12-31,600845.SH,宝信软件,2023-01-02 14:11:33,2023-01-02,2


In [24]:
m2m_model_dir = os.path.join(base_dir, "Models", "m2m100_1.2B")

print(f"Loading translation model from: {m2m_model_dir}")

m2m_tokenizer = AutoTokenizer.from_pretrained(m2m_model_dir)
m2m_model = AutoModelForSeq2SeqLM.from_pretrained(m2m_model_dir)

translator = pipeline(
    "translation",
    model=m2m_model,
    tokenizer=m2m_tokenizer,
    src_lang="zh",
    tgt_lang="en",
    device=0 if device == "cuda" else -1,
    truncation=True,
    max_length=512,
    num_beams=5,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    length_penalty=1.0
)

Loading translation model from: C:\Users\Bill Yin\Documents\GitHub\Strategy Development\NLP Equity L-S\Models\m2m100_1.2B


In [19]:
finbert_model = "yiyanghkust/finbert-tone"

print(f"Loading FinBERT model: {finbert_model}")

tokenizer = BertTokenizer.from_pretrained(finbert_model)
model = BertForSequenceClassification.from_pretrained(finbert_model).to(device)
model.eval()

Loading FinBERT model: yiyanghkust/finbert-tone


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30873, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [37]:
row = equity_reports.iloc[0]
ID = row["REPORT_ID"]
title_cn = row["TITLE"]
summary_cn = row["SUMMARY"]

# --- Run with timing ---
start = time.time()
translation_dict, logit_dict, softmax_dict = process_report(
    ID, title_cn, summary_cn, pre_processing=False
)
end = time.time()

# --- Print results ---
print("=== Original Chinese ===")
print("TITLE_CN :", title_cn)
print("SUMMARY_CN:", summary_cn)

print("\n=== Translated English ===")
print("TITLE_EN :", translation_dict["TITLE_EN"])
print("SUMMARY_EN:", translation_dict["SUMMARY_EN"])

print("\n=== Logit Dict ===")
for k, v in logit_dict.items():
    print(f"{k}: {v}")

print("\n=== Softmax Dict ===")
for k, v in softmax_dict.items():
    print(f"{k}: {v}")

print(f"\n✅ Completed in {end - start:.2f} seconds")

=== Original Chinese ===
TITLE_CN : 巴比食品（605338）：业绩目标稳健，完善激励机制
SUMMARY_CN: 巴比食品(605338)事件：公司发布《2022年限制性股票激励计划》，拟向激励对象授予的限制性股票数量为240.925万股，占本激励计划草案公告时公司股本总额24,800万股的0.97%。其中，首次授予220.925万股，占本激励计划公告时公司股本总额24,800万股的0.89%；预留20.00万股，占本激励计划公告时公司股本总额24,800万股的0.08%。点评：激励股份份额高，业绩目标设置稳健。公司此次限制性股份的激励份额占比近1%，激励对象涉及董事李俊、董秘/财务总监苏爽及中层管理人员及核心技术（业务）骨干等123人，覆盖激励对象范围广。授予价格15.51元/股，为公告前交易日均价的50%，激励力度大。考核目标以2022年的营收和扣非利润为基准，2023/2024/2025年相比2021年营收和扣非利润目标值为分别增长16%/32%/48%（对应同比为16%/13.8%/12.1%），触发值为分别增长12%/24%/36%（对应同比为12%/10.7%/9.7%），整体业绩目标设计稳健，进一步完善公司激励机制效果好。盈利预测与投资评级：展望后续，明年公司单店端有望迎来修复，南京工厂已投产，后续华北市场门店及团餐业务有望持续加速推进，华南市场开拓多年，单店提升和开店加速有望逐步呈现，华中市场收购项目稳步推进，单店后续有望逐步爬坡。巴比作为包子连锁头部品牌，有望受益餐饮供应链标准化提升的行业红利，团餐占比持续提升，门店端公司具备品牌和加盟连锁运营经验优势，长期开拓空间广阔。我们预计公司22-24年每股收益分别为0.83、1.04、1.33元，维持对公司的“买入”评级。风险因素：疫情反复、区域市场竞争加剧、原材料价格大幅上涨、食品安全问题。

=== Translated English ===
TITLE_EN : Barbie Food (605338): Stable performance goals and improved incentives
SUMMARY_EN: Barbie Food (605338) Event: The Company issued the 2022 Restrictive S